In [6]:
import sys

print(sys.executable)
print(sys.version)

c:\Program Files\Python312\python.exe
3.12.10 (tags/v3.12.10:0cc8128, Apr  8 2025, 12:21:36) [MSC v.1943 64 bit (AMD64)]


In [7]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

import operator
import os
from dotenv import load_dotenv

ModuleNotFoundError: No module named 'langgraph'

In [ ]:
os.environ["GROQ_API_KEY"] = "GROQ_API_KEY"

In [ ]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="Constructive feedback for the tweet.")
    score: int = Field(..., ge=0, le=5, description="Total score from rubric (0 to 5).")

In [ ]:
structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation)

In [ ]:
generator_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

evaluator_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

optimizer_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0
)

In [ ]:
# State
class TweetState(TypedDict):
    topic: str
    tweet: str
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str
    iteration: int
    max_iteration: int

tweet_history: Annotated[list[str], operator.add]
feedback_history: Annotated[list[str], operator.add]

In [ ]:
def generate_tweet(state: TweetState):

    # Prompt
    messages = [
        SystemMessage(
            content="You are a funny and clever Twitter/X influencer."
        ),
        HumanMessage(
            content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day-to-day English.
"""
        )
    ]

    # Generate tweet
    response = generator_llm.invoke(messages).content

    # Return updated state
    return { 'tweet': response, 'tweet_history': [response] }
    

In [ ]:
def evaluate_tweet(state: TweetState):

    # Prompt
    messages = [
        SystemMessage(
            content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets."
        ),
        HumanMessage(
            content=f"""
Evaluate the following tweet:

Tweet:
"{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality ⚙️ Is this fresh, or have you seen it a hundred times before?
2. Humor ⚙️ Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness ⚙️ Is it short, sharp, and scroll-stopping?
4. Virality Potential ⚙️ Would people retweet or share it?
5. Format ⚙️ Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format.
- It exceeds 280 characters.
- It reads like a traditional setup-punchline joke.
- Don't end with generic, throwaway, or deflating lines such as "Masterpieces".

Respond ONLY in this exact format:

evaluation: approved
feedback: <one paragraph>

OR

evaluation: needs_improvement
feedback: <one paragraph>
"""
        )
    ]

    # Evaluate the tweet
    response = evaluator_llm.invoke(messages).content

    # Parse the response
    lines = response.strip().split("\n")

    evaluation = "needs_improvement"
    feedback = response

    for line in lines:
        if line.lower().startswith("evaluation:"):
            evaluation = line.split(":", 1)[1].strip().lower()
        elif line.lower().startswith("feedback:"):
            feedback = line.split(":", 1)[1].strip()

    return {
        "evaluation": evaluation,
        "feedback": feedback,
    }
response = structured_evaluator_llm.invoke(messages)

return {'Evaluation':response.evaluation, 'feedback': response.feedback , 'feedback_history': [response.feedback]}

In [ ]:
def optimize_tweet(state: TweetState):
    messages = [
    SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
    HumanMessage(content=f"""\
    Improve the tweet based on this feedback:
    {state['feedback']}""")

    Topic: "{state['topic']}"
    Original Tweet:
    {state['tweet']}

    Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
"""
]
response = optimizer_llm.invoke(messages).content
iteration = state['iteration'] + 1

return {'tweet': response, 'iteration': iteration , 'tweet_history': [response]}

In [ ]:
def route_evaluation(state: TweetState):

    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
    return 'approved'
    else:
    return 'needs_improvement'

In [ ]:
graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')

graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'evaluate'})

workflow = graph.compile()

workflow

In [ ]:
initial_state = {
    "topid": "Egdbeq",
    "iteration": 1,
    "max_iterations": 5
}

workflow.invoke(initial_state)

: 